In [0]:
BASE = "/Volumes/workspace/default/pinterest_pipeline"
STAGING_PATH = f"{BASE}/staging"
INBOX_PATH = f"{BASE}/inbox"
dbutils.fs.mkdirs(INBOX_PATH)
CHECKPOINT_PATH = f"{BASE}/checkpoints/bronze_posts"

In [0]:
%sql
create volume if not exists workspace.default.pinterest_pipeline;

In [0]:
%sql

select
  *, 
  cast(year(date_posted) as smallint) as post_year, 
  cast(quarter(date_posted) as tinyint) as post_quarter 
from pinterest_posts
where date_posted IS NOT NULL;

In [0]:
# STAGING_PATH = f"{BASE}/staging"

from pyspark.sql import functions as F

df = (_sqldf
      .where(F.col("date_posted").isNotNull())
      .withColumn("_batch_year", F.col("post_year"))
      .withColumn("_batch_quarter", F.col("post_quarter"))
)

(df
    .write
    .format("json")
    .mode("overwrite")
    .partitionBy("post_year", "post_quarter")
    .save(STAGING_PATH)
)
